# EchoJEPA Training Pipeline Walkthrough

This notebook walks through every stage of the EchoJEPA training pipeline — from raw `.tar` shard on disk to the loss being computed — so you can visualize what the model actually sees and learns.

## Setup

In [ ]:
import sys
sys.path.insert(0, '../EchoJEPA')

import os
import io
import tarfile
import json
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import ListedColormap
from IPython.display import display, HTML

%matplotlib inline
plt.rcParams['figure.dpi'] = 120

---
## Stage 1 — Load Raw Frames from a WebDataset Shard

Each `.tar` shard contains `<uuid>.frames.npy` (uint8 RGB `(T, 336, 336, 3)`) and `<uuid>.metadata.json`.

**Source:** `src/datasets/webdataset_video_dataset.py:227-259`

In [ ]:
# === CONFIGURE: point to one of your shards ===
SHARD_PATH = "/hdd2/ahmedaly/preprocessed_by_alikhan_for_echojepa/shard-000000.tar"
# ================================================

def load_first_sample(shard_path):
    """Mimics WebDatasetVideoDataset._process_member()"""
    with tarfile.open(shard_path, "r:") as tar:
        for member in tar:
            if member.isfile() and member.name.endswith(".frames.npy"):
                raw = tar.extractfile(member)
                frames = np.load(io.BytesIO(raw.read()))  # (T, H, W, 3) uint8
                return frames, member.name
    return None, None

frames, sample_name = load_first_sample(SHARD_PATH)
print(f"Sample: {sample_name}")
print(f"Shape:  {frames.shape}  dtype: {frames.dtype}")
print(f"Value range: [{frames.min()}, {frames.max()}]")

In [ ]:
# Visualize a few raw frames
n_show = min(8, frames.shape[0])
fig, axes = plt.subplots(1, n_show, figsize=(2.5 * n_show, 2.5))
idxs = np.linspace(0, frames.shape[0] - 1, n_show, dtype=int)
for ax, i in zip(axes, idxs):
    ax.imshow(frames[i])
    ax.set_title(f"t={i}", fontsize=9)
    ax.axis("off")
fig.suptitle("Stage 1: Raw frames from .tar shard (uint8 RGB, 336×336)", fontsize=11)
plt.tight_layout()
plt.show()

---
## Stage 2 — Temporal Clip Sampling

From the full video `(T, H, W, 3)`, we sample a clip of `frames_per_clip=16` frames at `fps_sample=24` with `stride = fps_stored // fps_sample = 1`.

**Source:** `src/datasets/webdataset_video_dataset.py:261-288`

In [ ]:
def sample_clip(frames, frames_per_clip=16, fps_stored=24, fps_sample=24):
    """Mimics WebDatasetVideoDataset._sample_clip()"""
    stride = max(1, fps_stored // fps_sample)
    T = len(frames)
    fpc = frames_per_clip

    if T < fpc:
        # Video too short: interpolate indices
        idx = np.linspace(0, T - 1, fpc).astype(int)
    elif T < fpc * stride:
        # Video shorter than strided window: linspace
        idx = np.linspace(0, T - 1, fpc).astype(int)
    else:
        # Normal: pick random window of size fpc*stride, subsample with stride
        max_start = T - fpc * stride
        start = np.random.randint(0, max_start + 1)
        idx = np.arange(start, start + fpc * stride, stride)

    return frames[idx], idx

clip, clip_indices = sample_clip(frames, frames_per_clip=16)
print(f"Clip shape: {clip.shape}")
print(f"Frame indices selected: {clip_indices}")

In [ ]:
fig, axes = plt.subplots(2, 8, figsize=(18, 5))
for i in range(16):
    ax = axes[i // 8, i % 8]
    ax.imshow(clip[i])
    ax.set_title(f"t={clip_indices[i]}", fontsize=8)
    ax.axis("off")
fig.suptitle("Stage 2: Sampled clip — 16 frames @ 24fps", fontsize=11)
plt.tight_layout()
plt.show()

---
## Stage 3 — Spatial Augmentation & Normalization

The `VideoTransform` converts uint8 → float32, applies random resized crop (scale [0.5, 1.0], aspect [0.9, 1.1]), horizontal flip (50%), and normalizes to zero mean / unit variance.

**Source:** `app/vjepa/transforms.py:84-116`

In [ ]:
from app.vjepa.transforms import VideoTransform

# Build the transform used during pretraining
transform = VideoTransform(
    crop_size=336,
    random_resize_scale=(0.5, 1.0),
    random_resize_aspect_ratio=(0.9, 1.1),
    auto_augment=False,
    motion_shift=False,
    reprob=0.0,
)

# Apply to our clip (expects (T, H, W, C) uint8 numpy)
clip_tensor = transform(clip)  # returns (C, T, H, W) float32 tensor
print(f"After transform: {clip_tensor.shape}  dtype={clip_tensor.dtype}")
print(f"Value range: [{clip_tensor.min():.3f}, {clip_tensor.max():.3f}]")
print(f"Mean: {clip_tensor.mean():.3f}, Std: {clip_tensor.std():.3f}")

In [ ]:
# Visualize augmented clip (denormalize for display)
def denorm(t):
    """Undo normalization for visualization"""
    t = t.clone()
    t = t - t.min()
    t = t / (t.max() + 1e-8)
    return t

fig, axes = plt.subplots(2, 8, figsize=(18, 5))
for i in range(16):
    ax = axes[i // 8, i % 8]
    frame = denorm(clip_tensor[:, i]).permute(1, 2, 0).numpy()  # (H, W, C)
    ax.imshow(frame)
    ax.set_title(f"t={i}", fontsize=8)
    ax.axis("off")
fig.suptitle("Stage 3: After augmentation — random crop + normalize (C,T,H,W)", fontsize=11)
plt.tight_layout()
plt.show()

---
## Stage 4 — Patch Embedding (3D Convolution)

The ViT encoder first converts the `(C, T, H, W)` tensor into a sequence of patch tokens via a **3D convolution**:
- Kernel: `(tubelet_size=2, patch_size=16, patch_size=16)`
- Stride: same as kernel (non-overlapping)

This maps `(3, 16, 336, 336)` → `(D, 8, 21, 21)` → flattened to `(N=3528, D=1024)` tokens.

**Source:** `src/models/vision_transformer.py:161-213`

In [ ]:
# Compute the token grid dimensions
patch_size = 16
tubelet_size = 2
crop_size = 336
frames_per_clip = 16

T_tokens = frames_per_clip // tubelet_size   # 16 // 2 = 8
H_tokens = crop_size // patch_size            # 336 // 16 = 21
W_tokens = crop_size // patch_size            # 336 // 16 = 21
N_tokens = T_tokens * H_tokens * W_tokens     # 8 * 21 * 21 = 3528

print(f"Patch grid: T={T_tokens} × H={H_tokens} × W={W_tokens}")
print(f"Total tokens per clip: N = {N_tokens}")
print(f"Each token has D=1024 dimensions (ViT-L)")

In [ ]:
# Visualize the patch grid overlaid on one frame
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

frame_vis = denorm(clip_tensor[:, 0]).permute(1, 2, 0).numpy()

# Left: original frame
axes[0].imshow(frame_vis)
axes[0].set_title("Original frame", fontsize=10)
axes[0].axis("off")

# Right: with patch grid
axes[1].imshow(frame_vis)
for i in range(1, H_tokens):
    axes[1].axhline(y=i * patch_size, color='cyan', linewidth=0.5, alpha=0.6)
for j in range(1, W_tokens):
    axes[1].axvline(x=j * patch_size, color='cyan', linewidth=0.5, alpha=0.6)
axes[1].set_title(f"16×16 spatial patch grid ({H_tokens}×{W_tokens}={H_tokens*W_tokens} patches/frame)", fontsize=10)
axes[1].axis("off")

fig.suptitle("Stage 4: Patch embedding — 3D conv maps (3,16,336,336) → 3528 tokens of dim 1024", fontsize=11)
plt.tight_layout()
plt.show()

---
## Stage 5 — Masking: Context & Target Blocks

The `MaskCollator` generates two sets of masks:
- **Context mask (encoder input):** 8 small spatiotemporal blocks are removed → encoder sees the rest
- **Target mask (predictor must predict):** 2 large blocks (70% spatial, 100% temporal)

The encoder processes only the **unmasked (context)** patches. The predictor must predict the features at **target** positions.

**Source:** `src/masks/multiseq_multiblock3d.py:79-239`

In [ ]:
# Simulate the masking strategy from the config
np.random.seed(42)

def sample_block_mask(grid_t, grid_h, grid_w, t_scale, s_scale, aspect_ratio_range):
    """Sample a 3D block mask (1=keep, 0=masked)"""
    ar = np.random.uniform(*aspect_ratio_range)
    spatial_area = grid_h * grid_w * s_scale
    block_h = int(round(np.sqrt(spatial_area * ar)))
    block_w = int(round(np.sqrt(spatial_area / ar)))
    block_t = max(1, int(round(grid_t * t_scale)))
    block_h = min(block_h, grid_h)
    block_w = min(block_w, grid_w)
    block_t = min(block_t, grid_t)

    top = np.random.randint(0, grid_h - block_h + 1)
    left = np.random.randint(0, grid_w - block_w + 1)
    t_start = np.random.randint(0, grid_t - block_t + 1)

    mask = np.ones((grid_t, grid_h, grid_w), dtype=np.int32)
    mask[t_start:t_start+block_t, top:top+block_h, left:left+block_w] = 0
    return mask

# Context mask: start with all 1s, multiply by 8 small block removals
context_mask = np.ones((T_tokens, H_tokens, W_tokens), dtype=np.int32)
for _ in range(8):  # npred=8 context blocks
    block = sample_block_mask(T_tokens, H_tokens, W_tokens,
                               t_scale=1.0, s_scale=0.15,
                               aspect_ratio_range=(0.75, 1.5))
    context_mask *= block

# Target mask: 2 large blocks
target_mask = np.zeros((T_tokens, H_tokens, W_tokens), dtype=np.int32)
for _ in range(2):  # 2 large target blocks
    block = sample_block_mask(T_tokens, H_tokens, W_tokens,
                               t_scale=1.0, s_scale=0.70,
                               aspect_ratio_range=(0.75, 1.5))
    target_mask = np.maximum(target_mask, 1 - block)  # Union of masked regions

context_indices = np.argwhere(context_mask.flatten() == 1).squeeze()
target_indices = np.argwhere(target_mask.flatten() == 1).squeeze()

print(f"Total tokens: {N_tokens}")
print(f"Context (encoder sees): {len(context_indices)} tokens ({100*len(context_indices)/N_tokens:.1f}%)")
print(f"Target (predictor predicts): {len(target_indices)} tokens ({100*len(target_indices)/N_tokens:.1f}%)")

In [ ]:
# Visualize masks across temporal slices
fig, axes = plt.subplots(3, T_tokens, figsize=(2.5 * T_tokens, 7.5))

cmap_ctx = ListedColormap(['#ff4444', '#44cc44'])  # red=masked, green=visible
cmap_tgt = ListedColormap(['#dddddd', '#4488ff'])  # gray=not target, blue=target

for t in range(T_tokens):
    # Row 0: augmented frame (average of tubelet)
    t_start_frame = t * tubelet_size
    t_end_frame = min(t_start_frame + tubelet_size, 16)
    avg_frame = denorm(clip_tensor[:, t_start_frame:t_end_frame].mean(dim=1)).permute(1, 2, 0).numpy()
    axes[0, t].imshow(avg_frame)
    axes[0, t].set_title(f"t={t}", fontsize=8)
    axes[0, t].axis("off")

    # Row 1: context mask
    axes[1, t].imshow(context_mask[t], cmap=cmap_ctx, vmin=0, vmax=1, interpolation='nearest')
    axes[1, t].axis("off")

    # Row 2: target mask
    axes[2, t].imshow(target_mask[t], cmap=cmap_tgt, vmin=0, vmax=1, interpolation='nearest')
    axes[2, t].axis("off")

axes[0, 0].set_ylabel("Frame", fontsize=9, rotation=0, labelpad=40)
axes[1, 0].set_ylabel("Context\n(green=visible)", fontsize=9, rotation=0, labelpad=55)
axes[2, 0].set_ylabel("Target\n(blue=predict)", fontsize=9, rotation=0, labelpad=55)

fig.suptitle("Stage 5: Masking — 8 small context blocks removed, 2 large target blocks to predict", fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# Visualize what the encoder actually sees (masked input at frame level)
fig, axes = plt.subplots(2, 8, figsize=(18, 5))
for i in range(16):
    ax = axes[i // 8, i % 8]
    frame = denorm(clip_tensor[:, i]).permute(1, 2, 0).numpy().copy()
    t_idx = i // tubelet_size
    # Darken masked patches
    for ph in range(H_tokens):
        for pw in range(W_tokens):
            if context_mask[t_idx, ph, pw] == 0:  # masked out
                y0, y1 = ph * patch_size, (ph + 1) * patch_size
                x0, x1 = pw * patch_size, (pw + 1) * patch_size
                frame[y0:y1, x0:x1] *= 0.15
    ax.imshow(frame)
    ax.set_title(f"t={i}", fontsize=8)
    ax.axis("off")
fig.suptitle("What the ENCODER sees (darkened patches are masked out — not fed to encoder)", fontsize=11)
plt.tight_layout()
plt.show()

---
## Stage 6 — Forward Pass: Encoder → Predictor → Loss

This is the core of V-JEPA training. The diagram below shows the full forward pass.

```
                  ┌─────────────────────────────────────┐
                  │        Input: (B, 3, 16, 336, 336)  │
                  └─────────────┬───────────────────────┘
                                │
              ┌─────────────────┼─────────────────┐
              ▼                                   ▼
   ┌──────────────────┐              ┌──────────────────┐
   │  Online Encoder   │              │  Target Encoder   │
   │  (ViT-L, trained) │              │  (EMA copy, no    │
   │                   │              │   gradients)       │
   │  Input: context   │              │  Input: ALL tokens │
   │  patches only     │              │  (no masking)      │
   └────────┬──────────┘              └────────┬──────────┘
            │                                  │
            │ z_context (B, K_ctx, 1024)       │ h_all (B, N, 1024)
            │                                  │
            ▼                                  │
   ┌──────────────────┐                        │
   │   Predictor       │                        │
   │   (12 blocks,     │                        │
   │    dim=384)       │                        │
   │                   │                        │
   │  Input: z_context │                        │
   │  + mask_tokens    │                        │
   │  at target pos    │                        │
   └────────┬──────────┘                        │
            │                                   │
            │ z_pred (B, K_tgt, 1024)           │ h_target = h_all[target_idx]
            │                                   │   (B, K_tgt, 1024)
            ▼                                   ▼
   ┌─────────────────────────────────────────────────┐
   │           Loss = mean(|z_pred - h_target|^p) / p │
   │           (L2 loss in latent space, no pixels!)   │
   └───────────────────────────────────────────────────┘
                                │
                                ▼
                     Backprop through encoder
                     + predictor only.
                     Target encoder updated via EMA.
```

### 6a. Encoder Forward Pass

The online encoder receives **only context patches** (not masked ones). The target encoder receives **all patches**.

**Source:** `src/models/vision_transformer.py:161-213`

```python
# -- app/vjepa/train.py:593-631 (simplified)
def forward_target():
    with torch.no_grad():
        h = target_encoder(clips)           # ALL patches → (B, 3528, 1024)
        h = F.layer_norm(h, (h.size(-1),))  # Normalize targets
    return h

def forward_context():
    z = encoder(clips, masks=masks_enc)      # context patches only → (B, K_ctx, 1024)
    z = predictor(z, masks_enc, masks_pred)   # predict target features → (B, K_tgt, 1024)
    return z
```

In [ ]:
# Simulate the token flow (dimensions only — no actual model weights needed)
B = 1  # batch size
D = 1024  # ViT-L embed dim
K_ctx = len(context_indices)
K_tgt = len(target_indices)

print("=== Encoder ===")
print(f"  Input clips:        ({B}, 3, 16, 336, 336)")
print(f"  After patch embed:  ({B}, {N_tokens}, {D})   [3D conv]")
print(f"  After masking:      ({B}, {K_ctx}, {D})   [keep context only]")
print(f"  Encoder output z:   ({B}, {K_ctx}, {D})   [24 transformer blocks]")
print()
print("=== Target Encoder (EMA, no grad) ===")
print(f"  Input clips:        ({B}, 3, 16, 336, 336)")
print(f"  Target output h:    ({B}, {N_tokens}, {D})   [all patches, layer-normed]")
print(f"  h at target idx:    ({B}, {K_tgt}, {D})   [extract target positions]")
print()
print("=== Predictor ===")
print(f"  Input z_context:    ({B}, {K_ctx}, 384)   [projected to predictor dim]")
print(f"  + mask tokens:      ({B}, {K_tgt}, 384)   [learnable tokens at target positions]")
print(f"  Concatenated:       ({B}, {K_ctx + K_tgt}, 384)   [sorted by position]")
print(f"  After 12 blocks:    ({B}, {K_ctx + K_tgt}, 384)")
print(f"  Predictor output:   ({B}, {K_tgt}, {D})   [extract target predictions, project to D]")

### 6b. Predictor Detail

The predictor is the key component. It takes context tokens from the encoder and must **predict the features at target positions**.

**Source:** `src/models/predictor.py:166-246`

```python
def forward(self, x, masks_x, masks_y, mask_index=1):
    # x: context tokens from encoder  (B, K_ctx, D_enc)

    # 1. Project to predictor dimension
    x = self.predictor_embed(x)               # (B, K_ctx, 384)

    # 2. Add positional embeddings to context tokens
    x += apply_masks(self.predictor_pos_embed, masks_x)

    # 3. Create learnable mask tokens at target positions
    pred_tokens = self.mask_tokens[mask_index]  # (1, 1, 384)
    pred_tokens = pred_tokens.expand(B, K_tgt, 384)
    pred_tokens += apply_masks(self.predictor_pos_embed, masks_y)

    # 4. Concatenate context + target, sort by grid position
    x = torch.cat([x, pred_tokens], dim=1)     # (B, K_ctx + K_tgt, 384)

    # 5. Self-attention through 12 predictor blocks
    for blk in self.predictor_blocks:
        x = blk(x)

    # 6. Extract target predictions, project back
    x = x[:, K_ctx:]                           # (B, K_tgt, 384)
    x = self.predictor_proj(x)                 # (B, K_tgt, 1024)
    return x
```

### 6c. Loss Function

The loss is the **L_p norm** between predicted features and target features, computed **in latent space** (not pixel space!).

**Source:** `app/vjepa/train.py:593-601`

```python
def loss_fn(z, h):
    """z: predictor outputs, h: target encoder outputs"""
    loss = torch.mean(torch.abs(z - h) ** loss_exp) / loss_exp
    return loss
```

With `loss_exp=1.0` (from config), this is simply **L1 (mean absolute error)** in the 1024-dim feature space.

In [ ]:
# Simulate the loss computation
loss_exp = 1.0  # from config

# Fake features for illustration
z_pred = torch.randn(B, K_tgt, D)   # predictor output
h_target = torch.randn(B, K_tgt, D)  # target encoder output (normalized)

# JEPA loss: L_p in latent space
loss = torch.mean(torch.abs(z_pred - h_target) ** loss_exp) / loss_exp

print(f"Predictor output shape:  {z_pred.shape}")
print(f"Target features shape:   {h_target.shape}")
print(f"Loss (L{loss_exp:.0f}):              {loss.item():.4f}")
print()
print("Key insight: the model never reconstructs pixels!")
print("It learns to predict abstract features in the target encoder's representation space.")

---
## Stage 7 — EMA Update & Optimization

After each step:
1. **Backprop** through encoder + predictor (target encoder has no gradients)
2. **AdamW** updates encoder and predictor parameters
3. **EMA** updates target encoder: `θ_target = m · θ_target + (1-m) · θ_encoder`

**Source:** `app/vjepa/train.py:622-631`

```python
# EMA update (after optimizer step)
m = next(momentum_scheduler)  # e.g., 0.99925
with torch.no_grad():
    torch._foreach_mul_(target_params, m)            # target *= m
    torch._foreach_add_(target_params, online_params, alpha=1 - m)  # target += (1-m)*online
```

In [ ]:
# Visualize EMA momentum schedule
ema_start, ema_end = 0.99925, 0.99925  # constant in this config
ipe = 1000  # iterations per epoch
num_epochs = 200
total_steps = ipe * num_epochs

steps = np.arange(total_steps)
momentum = ema_start + steps * (ema_end - ema_start) / total_steps

# LR schedule: warmup + cosine
warmup_epochs = 40
start_lr = 4e-6
ref_lr = 1.64e-5
final_lr = 1.64e-5

lr_schedule = np.zeros(total_steps)
warmup_steps = warmup_epochs * ipe
# Linear warmup
lr_schedule[:warmup_steps] = np.linspace(start_lr, ref_lr, warmup_steps)
# Cosine decay
remaining = total_steps - warmup_steps
lr_schedule[warmup_steps:] = final_lr + 0.5 * (ref_lr - final_lr) * (1 + np.cos(np.pi * np.arange(remaining) / remaining))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

ax1.plot(steps / ipe, lr_schedule * 1e5, color='#4488ff')
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Learning Rate (×1e-5)")
ax1.set_title("LR Schedule: 40-epoch warmup → cosine")
ax1.axvline(x=warmup_epochs, color='red', linestyle='--', alpha=0.5, label='warmup end')
ax1.legend()

ax2.plot(steps / ipe, momentum, color='#44cc44')
ax2.set_xlabel("Epoch")
ax2.set_ylabel("EMA Momentum")
ax2.set_title(f"EMA Momentum (constant {ema_start})")
ax2.set_ylim(0.998, 1.001)

fig.suptitle("Stage 7: Optimization schedules", fontsize=11)
plt.tight_layout()
plt.show()

---
## Summary: End-to-End Training Step

| Step | What happens | Shape | Source |
|------|-------------|-------|--------|
| 1. Load | `.npy` from `.tar` shard | `(T, 336, 336, 3)` uint8 | `webdataset_video_dataset.py:227` |
| 2. Clip | Sample 16 frames at 24fps | `(16, 336, 336, 3)` uint8 | `webdataset_video_dataset.py:261` |
| 3. Augment | Random crop + normalize | `(3, 16, 336, 336)` float32 | `transforms.py:84` |
| 4. Patch | 3D conv → tokens | `(3528, 1024)` | `vision_transformer.py:161` |
| 5. Mask | Context/target split | `(K_ctx, 1024)` / `(K_tgt, 1024)` | `multiseq_multiblock3d.py:172` |
| 6a. Encode | ViT-L on context patches | `(K_ctx, 1024)` | `vision_transformer.py:161` |
| 6b. Predict | Predict target features | `(K_tgt, 1024)` | `predictor.py:166` |
| 6c. Target | EMA encoder on all patches | `(3528, 1024)` → `(K_tgt, 1024)` | `train.py:593` |
| 6d. Loss | L1 in latent space | scalar | `train.py:597` |
| 7. Update | AdamW + EMA | — | `train.py:622` |

---
## Quick Reference: Key Source Files

| Component | File | Key Lines |
|-----------|------|-----------|
| Training loop | `app/vjepa/train.py` | 496-763 |
| Data loading | `src/datasets/webdataset_video_dataset.py` | 105-289 |
| Transforms | `app/vjepa/transforms.py` | 84-116 |
| Masking | `src/masks/multiseq_multiblock3d.py` | 79-239 |
| Encoder (ViT) | `src/models/vision_transformer.py` | 161-213 |
| Predictor | `src/models/predictor.py` | 166-246 |
| Mask application | `src/masks/utils.py` | 9-21 |
| Loss function | `app/vjepa/train.py` | 593-601 |
| EMA update | `app/vjepa/train.py` | 622-631 |
| LR/WD schedulers | `src/utils/schedulers.py` | — |
| Config | `training/pretrain_icardio_336px_16f.yaml` | — |